# 02 - Train RMIA reference models

Stage 4: 32 shadow models, each on a random half of the training set, so every example is absent from roughly 16 of them. They are condition-independent -- the same 32 serve all eight forget conditions -- and nothing blocks on them until the privacy audit, so they can run early on a spare account.

**Set `ACCOUNT` below** to this account's number. Every account runs the identical
notebook with a different `ACCOUNT`, and they compute disjoint slices of the work with
no coordination -- run identifiers are pure functions of the config, so all accounts
derive the same work list and each takes its own stripe. Running everything on one
account also works: set `OF = 1` and the slice becomes the whole list.

Runs already present in the attached artefact dataset are skipped, so a session that
dies costs only its in-flight model.


In [ ]:
# --- clone the repo at a PINNED commit ------------------------------------------------------
# Clone rather than `pip install git+...`. A wheel would contain only src/forgetcheck/, but the
# CLI also needs configs/ (the metric registry, seed streams, audit protocols) and
# data/memorization/ (the RUM scores -- 400 KB, committed precisely so a fresh session does not
# have to re-download 2 GB from Google Drive).
#
# Pinning is what makes provenance work: every record this session writes carries this commit,
# so any result can be traced back to the exact code that produced it.
REPO   = "https://github.com/hyperreal2005/Minor-Project.git"
COMMIT = "main"          # <-- pin to a sha for real runs, e.g. "a1b2c3d"

import os
from pathlib import Path

os.chdir("/kaggle/working")
if not Path("Minor-Project").exists():
    !git clone --quiet $REPO
%cd /kaggle/working/Minor-Project
!git fetch --quiet --all && git checkout --quiet $COMMIT
!git log -1 --format="pinned at %h  %s"
!pip install -q -e .


In [ ]:
import importlib
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/Minor-Project")

# Make the package importable *in this kernel*.
#
# `pip install -e .` writes a .pth file into site-packages, and .pth files are only processed at
# interpreter startup. The kernel was already running when the previous cell installed, so
# sys.path never picked it up and `import forgetcheck` fails with ModuleNotFoundError. The
# `!forgetcheck` CLI calls below are unaffected -- each spawns a fresh Python that does read the
# .pth -- which makes this failure look stranger than it is.
#
# Adding src/ directly is deterministic and avoids making anyone restart the kernel.
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

# `!` cells run in a subshell, which inherits this. Belt and braces: the console script should
# already be on PATH after the editable install, but if it is not, PYTHONPATH keeps
# `python -m forgetcheck.cli` working as a fallback.
import os
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")

import forgetcheck
print("forgetcheck imported from:", Path(forgetcheck.__file__).parent)

import shutil

# CIFAR-10 downloads at 100-130 kB/s on Kaggle -- 20 to 30 minutes, repeated on every session and
# every account. Attaching it as a Dataset (Add Input -> Datasets) skips that entirely:
# torchvision checks the md5s of the extracted folder and only downloads if it is missing or
# corrupt. 00_verify_setup.ipynb has a cell that creates the dataset once.
#
# Kaggle's mount layout varies with how a dataset was uploaded -- it may sit at
# /kaggle/input/<slug>/, or nested as /kaggle/input/datasets/<user>/<slug>/, or one level deeper
# again if the dataset was created from a notebook's output directory. So SEARCH for the folder
# rather than assume a path (`**/` matches at any depth, including directly under /kaggle/input),
# and then verify the copy actually landed. An earlier version of this cell
# used `cp ... 2>/dev/null || true` followed by an unconditional success message: a failed copy
# reported success and CIFAR silently re-downloaded anyway, costing ~28 minutes while the output
# claimed otherwise. Never report an outcome that was not checked.
def _restore(name, dest):
    "Find directory `name` anywhere under /kaggle/input and copy it to `dest`."
    dest = Path(dest)
    if dest.is_dir():
        print(f"{name}: already present")
        return True
    found = sorted(Path("/kaggle/input").glob(f"**/{name}"))
    if not found:
        return False
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(found[0], dest, dirs_exist_ok=True)
    print(f"{name}: copied from {found[0]}")
    return True

# Restore CIFAR-10 by locating its *contents*, not its folder name.
#
# Kaggle does not necessarily preserve the directory that was uploaded: the batches may end up
# inside `cifar-10-batches-py/`, or flattened straight to the dataset root. Searching for the
# folder name therefore reports "not found" while the data sits one level up in plain view --
# which is exactly what happened here, and cost a 25-minute re-download.
#
# So anchor on a file that must exist (`test_batch`) and take whatever directory contains it.
# That handles both layouts, and any future one.
def _restore_cifar(dest):
    dest = Path(dest)
    if dest.is_dir() and len(list(dest.glob("*_batch*"))) == 6:
        print("cifar-10: already present")
        return True
    hits = sorted(Path("/kaggle/input").glob("**/test_batch"))
    if not hits:
        return False
    src = hits[0].parent
    dest.mkdir(parents=True, exist_ok=True)
    for p in src.iterdir():
        if p.is_file():
            shutil.copy2(p, dest / p.name)
    print(f"cifar-10: copied from {src}")
    return True

CIFAR_DEST = REPO_DIR / "data" / "cifar-10-batches-py"
if not _restore_cifar(CIFAR_DEST):
    print("!! no CIFAR-10 batches found under /kaggle/input -- it will DOWNLOAD (~25 min)")
    print("!! attached:", [p.name for p in sorted(Path("/kaggle/input").glob("*"))] or "(none)")

# Verify rather than trust: torchvision needs 5 training batches plus test_batch.
if CIFAR_DEST.is_dir():
    n = len(list(CIFAR_DEST.glob("*_batch*")))
    print(f"   {n}/6 batch files{'' if n == 6 else '  <-- INCOMPLETE, will re-download'}")

# Previous artefacts, so runs another session already finished are skipped rather than repeated.
for _name in ("artifacts", "results"):
    if not _restore(_name, REPO_DIR / _name):
        print(f"{_name}: none attached - starting fresh")

import torch
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("!! running on CPU. Set Settings -> Accelerator -> GPU.")
    print("!! On CPU one 30-epoch training run takes ~4.8 hours instead of ~12 minutes.")

CLI = "forgetcheck" if shutil.which("forgetcheck") else f"{sys.executable} -m forgetcheck.cli"
print("cli:", CLI)


In [ ]:
ACCOUNT = 1     # <-- this account's number, 1-based
OF      = 3     # <-- how many accounts share this stage (1 takes everything)

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
STAGE  = 4
print(f"account {ACCOUNT} of {OF}, stage {STAGE}, device {DEVICE}")


## What this account will do

In [ ]:
!{CLI} --root . --dry-run queue --stage {STAGE} --account {ACCOUNT} --of {OF}


## Run it

The long cell. It prints each run as it starts and finishes, so you can watch progress and estimate the time remaining.

In [ ]:
!{CLI} --root . --device {DEVICE} queue --stage {STAGE} --account {ACCOUNT} --of {OF}


## Inspect what came out

In [ ]:
from forgetcheck.registry import read_records
import pandas as pd

df = read_records("results/records")
print(f"{len(df)} rows across {df['run_id'].nunique()} runs\n")

# Pivot on (metric, probe_set), NOT metric alone. macro_f1 and ce_loss are each recorded for
# more than one probe set, so collapsing on metric silently averages two different quantities
# into one plausible-looking number.
wide = df.pivot_table(
    index=["run_id", "role"], columns=["metric", "probe_set"], values="value"
)
wide.round(4)


### What to look for

Shadow models have **no forget set**, so there is no `forget_acc` here and `forget_id` reads
`full` for every row. That is correct, not a gap.

* **`test_acc` around 0.89**, roughly 3-4 pp below the full-data models. Each shadow trains on a
  random *half* of the training set, so it should be measurably weaker. If they matched the
  full-data models, the subsetting would not be happening.
* **Runtime around half** that of a Stage 3 run, for the same reason.
* **OUT coverage** is the property the whole privacy audit rests on, and the check below measures
  it: every target example must be *absent* from a decent number of shadows, because those are
  the models RMIA compares a target against. With 32 shadows at 50% each, expect a mean near 16.
  A low minimum would mean some examples have almost no reference distribution, making their
  per-example attack scores unreliable.


In [ ]:
# OUT coverage: for each example, how many shadows did NOT train on it? Those are the reference
# models RMIA scores a target against, so this is the property that makes the attack possible.
import numpy as np
from pathlib import Path

from forgetcheck.config import Context, find_configs
from forgetcheck.train import shadow_indices

ctx = Context(configs=find_configs(), root=Path("."))
n_train = ctx.base["dataset"]["n_train"]
total = ctx.base["shadows"]["count"]

have = sorted(
    int(r.rsplit("shadow", 1)[-1]) for r in ctx.store.iter_checkpoints(role="shadow")
)
print(f"{len(have)}/{total} shadows trained so far")

if have:
    subsets = [
        set(shadow_indices(n_train, i, audit_seed=ctx.seeds["audit"]).tolist()) for i in have
    ]
    sample = np.random.default_rng(0).choice(n_train, 1000, replace=False)
    out = np.array([sum(int(x) not in sub for sub in subsets) for x in sample])
    print(f"OUT coverage over 1000 random examples: mean {out.mean():.1f}, "
          f"min {out.min()}, max {out.max()} (of {len(have)} shadows)")
    if len(have) == total:
        print(f"   expected mean ~{total // 2}; a low minimum means some examples have too few "
              "reference models for a reliable per-example score")


In [ ]:
!{CLI} --root . status


In [ ]:
# --- push the artefacts back out -------------------------------------------------------------
# Kaggle sessions are disposable. Anything not saved to a Dataset version is gone, and the next
# session would recompute it. /kaggle/working persists per notebook (~20 GB); a Dataset is what
# shares it between notebooks and accounts.
import json
from pathlib import Path

import shutil

OUT = Path("/kaggle/working/to_upload")
OUT.mkdir(exist_ok=True)

# Copy and verify. The same `cp ... 2>/dev/null || true` pattern that lived here once produced a
# CIFAR dataset containing nothing but its metadata file, and said so to nobody.
staged = {}
for name in ("artifacts", "results"):
    src = Path("/kaggle/working/Minor-Project") / name
    if src.is_dir():
        shutil.copytree(src, OUT / name, dirs_exist_ok=True)
        staged[name] = sum(1 for p in (OUT / name).rglob("*") if p.is_file())
    else:
        staged[name] = 0

for name, n in staged.items():
    print(f"{name}: {n} files staged{'  <-- nothing to upload' if n == 0 else ''}")
if not staged.get("artifacts"):
    raise SystemExit("no artifacts to upload; did the queue cell actually run?")

META = OUT / "dataset-metadata.json"
META.write_text(json.dumps({
    "title": "forgetcheck-artifacts",
    "id": "YOUR-KAGGLE-USERNAME/forgetcheck-artifacts",   # <-- your username
    "licenses": [{"name": "CC0-1.0"}],
}, indent=2))

# Needs an API token at ~/.kaggle/kaggle.json (Kaggle -> Account -> Create New API Token).
# First time:   !kaggle datasets create  -p $OUT --dir-mode zip
# Afterwards:   !kaggle datasets version -p $OUT -m "stage N account K" --dir-mode zip
#
# Simplest alternative: just "Save Version" the notebook. /kaggle/working persists per notebook
# (~20 GB), which is enough for one account to resume itself -- but a Dataset is what shares
# artefacts between accounts, and that is what the team needs.
print(f"staged in {OUT} - uncomment whichever line above applies")
